# Learning Objectives
In this notebook, you will learn Spark Dataframe APIs.

# Question List

Solve the following questions using Spark Dataframe APIs

### Join

1. easy - https://pgexercises.com/questions/joins/simplejoin.html
2. easy - https://pgexercises.com/questions/joins/simplejoin2.html
3. easy - https://pgexercises.com/questions/joins/self2.html 
4. medium - https://pgexercises.com/questions/joins/threejoin.html (three join)
5. medium - https://pgexercises.com/questions/joins/sub.html (subquery and join)

### Aggregation

1. easy - https://pgexercises.com/questions/aggregates/count3.html Group by order by
2. easy - https://pgexercises.com/questions/aggregates/fachours.html group by order by
3. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth.html group by with condition 
4. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth2.html group by multi col
5. easy - https://pgexercises.com/questions/aggregates/members1.html count distinct
6. med - https://pgexercises.com/questions/aggregates/nbooking.html group by multiple cols, join

### String & Date

1. easy - https://pgexercises.com/questions/string/concat.html format string
2. easy - https://pgexercises.com/questions/string/case.html WHERE + string function
3. easy - https://pgexercises.com/questions/string/reg.html WHERE + string function
4. easy - https://pgexercises.com/questions/string/substr.html group by, substr
5. easy - https://pgexercises.com/questions/date/series.html generate ts
6. easy - https://pgexercises.com/questions/date/bookingspermonth.html extract month from ts

### Question

How can you produce a list of the start times for bookings by members named 'David Farrell'?

https://pgexercises.com/questions/joins/simplejoin.html

In [0]:
# Write you solution here
# hint: you might need to re-run `0 - ETL pgexercieses CSV files` notebook to init tables
members = spark.table("members")

df = spark.sql("select * from members where members.firstname = 'David'")
display(df)

memid,firstname,surname,address,zipcode,telephone,recommendedby,joindate


### Question 1
How can you produce a list of the start times for bookings by members named 'David Farrell'?



In [0]:
from pyspark.sql import functions as F

# searches for the table physically named bookings and members (in catalog)
bookings = spark.table("bookings")
members = spark.table("members")

# Combining bookings with members table on memid field, then filteres for first and last name. .select returns only that column
results = bookings.join(members, bookings.memid == members.memid) \
    .filter((members.firstname == "David") & (members.surname == "Farrell")) \
    .select(bookings.starttime)
    
  
display(results)

starttime


### Question 2
How can you produce a list of the start times for bookings for tennis courts, for the date '2012-09-21'? Return a list of start time and facility name pairings, ordered by the time.


In [0]:
from pyspark.sql import functions as F

# Searches for 3 tables
bookings = spark.table("bookings")
members = spark.table("members")
facilities = spark.table("facilities")

# Joining on facid and filtering for tennis court
#F.col(): Identifies a specific column in a specific table (like bks.facid)
result = bookings.alias("bks") \
    .join(facilities.alias("fas"), F.col("bks.facid") == F.col("fas.facid")) \
    .filter(
        (F.col("bks.starttime") >= "2012-09-21") & 
        (F.col("bks.starttime") < "2012-09-22") & 
        (F.col("fas.name").contains("Tennis Court"))
    ) \
    .select(
        F.col("bks.starttime").alias("start"), 
        F.col("fas.name")
    ) \
    .orderBy("start")

# To see your results:
result.show()

+-------------------+--------------+
|              start|          name|
+-------------------+--------------+
|2012-09-21 08:00:00|Tennis Court 2|
|2012-09-21 08:00:00|Tennis Court 1|
|2012-09-21 09:30:00|Tennis Court 1|
|2012-09-21 10:00:00|Tennis Court 2|
|2012-09-21 11:30:00|Tennis Court 2|
|2012-09-21 12:00:00|Tennis Court 1|
|2012-09-21 13:30:00|Tennis Court 1|
|2012-09-21 14:00:00|Tennis Court 2|
|2012-09-21 15:30:00|Tennis Court 1|
|2012-09-21 16:00:00|Tennis Court 2|
|2012-09-21 17:00:00|Tennis Court 1|
|2012-09-21 18:00:00|Tennis Court 2|
+-------------------+--------------+



### Question 3 
How can you output a list of all members, including the individual who recommended them (if any)? Ensure that results are ordered by (surname, firstname).



In [0]:
from pyspark.sql import functions as F

# We alias the same DataFrame twice: once as 'mems' and once as 'recs'
mems = members.alias("mems")
recs = members.alias("recs")

# Perform a LEFT JOIN (to include members who weren't recommended by anyone)
result = mems.join(
    recs, 
    F.col("mems.recommendedby") == F.col("recs.memid"), 
    how="left"
) \
.select(
    F.col("mems.firstname").alias("memfname"),
    F.col("mems.surname").alias("memsname"),
    F.col("recs.firstname").alias("recfname"),
    F.col("recs.surname").alias("recsname")
) \
.orderBy("memsname", "memfname")

# Display the result
result.show()

+-----------------+---------+--------+--------+
|         memfname| memsname|recfname|recsname|
+-----------------+---------+--------+--------+
|        Mackenzie|     Anna|   Smith|  Darren|
|            Baker|     Anne|Stibbons|  Ponder|
|            Tracy|   Burton|    NULL|    NULL|
|             Owen|  Charles|   Smith|  Darren|
|            Smith|   Darren|    NULL|    NULL|
|            Smith|   Darren|    NULL|    NULL|
|          Farrell|    David|    NULL|    NULL|
|            Jones|    David|Joplette|  Janice|
|           Pinker|    David| Farrell|  Jemima|
|            Jones|  Douglas|   Jones|   David|
|          Crumpet|    Erica|   Smith|   Tracy|
|            Bader| Florence|Stibbons|  Ponder|
|            GUEST|    GUEST|    NULL|    NULL|
|          Butters|   Gerald|   Smith|  Darren|
|           Rumney|Henrietta| Genting| Matthew|
|Worthington-Smyth|    Henry|   Smith|   Tracy|
|       Tupperware| Hyacinth|    NULL|    NULL|
|            Smith|     Jack|   Smith|  

### Question 4
How can you produce a list of all members who have used a tennis court? Include in your output the name of the court, and the name of the member formatted as a single column. Ensure no duplicate data, and order by the member name followed by the facility name.

In [0]:
from pyspark.sql import functions as F

# Join members, bookings, and facilities
result = members.alias("mems") \
    .join(bookings.alias("bks"), F.col("mems.memid") == F.col("bks.memid")) \
    .join(facilities.alias("facs"), F.col("bks.facid") == F.col("facs.facid")) \
    .filter(F.col("facs.name").contains("Tennis Court")) \
    .select(
        # Combine firstname and surname with a space (ws stands for with separator)
        F.concat_ws(" ", F.col("mems.firstname"), F.col("mems.surname")).alias("member"),
        F.col("facs.name").alias("facility")
    ) \
    .distinct() \
    .orderBy("member", "facility")

result.show()

+--------------+--------------+
|        member|      facility|
+--------------+--------------+
|Bader Florence|Tennis Court 1|
|Bader Florence|Tennis Court 2|
|    Baker Anne|Tennis Court 1|
|    Baker Anne|Tennis Court 2|
| Baker Timothy|Tennis Court 1|
| Baker Timothy|Tennis Court 2|
|    Boothe Tim|Tennis Court 1|
|    Boothe Tim|Tennis Court 2|
|Butters Gerald|Tennis Court 1|
|Butters Gerald|Tennis Court 2|
|   Coplin Joan|Tennis Court 1|
| Crumpet Erica|Tennis Court 1|
|    Dare Nancy|Tennis Court 1|
|    Dare Nancy|Tennis Court 2|
| Farrell David|Tennis Court 1|
| Farrell David|Tennis Court 2|
|Farrell Jemima|Tennis Court 1|
|Farrell Jemima|Tennis Court 2|
|   GUEST GUEST|Tennis Court 1|
|   GUEST GUEST|Tennis Court 2|
+--------------+--------------+
only showing top 20 rows


### Question 5
How can you output a list of all members, including the individual who recommended them (if any), without using any joins? Ensure that there are no duplicates in the list, and that each firstname + surname pairing is formatted as a column and ordered.


In [0]:
from pyspark.sql import functions as F

members = spark.table("members")

# Creating a version of the table. It contains only two things: the ID and the Full Name (formatted with a space). We call this recommender because, in the context of this join, these are the people who gave the recommendations.
recs = members.select(
    F.col("memid"), 
    F.concat_ws(" ", "firstname", "surname").alias("recommender")
)

# members table again, but this time we are treating them as the new members who were referred
# broadcast tells it to use the table we created above  
# Have used member id as well because it was not showing Smith Darren twice, even though they have different ids
result = members.alias("mems") \
    .select(
        F.col("memid").alias("member_id"),
        F.concat_ws(" ", "firstname", "surname").alias("member"),
        "recommendedby"
    ) \
    .join(F.broadcast(recs), F.col("recommendedby") == F.col("memid"), "left") \
    .select("member_id", "member", "recommender") \
    .distinct() \
    .orderBy("member")

display(result)

member_id,member,recommender
15,Bader Florence,Stibbons Ponder
12,Baker Anne,Stibbons Ponder
16,Baker Timothy,Farrell Jemima
8,Boothe Tim,Rownam Tim
5,Butters Gerald,Smith Darren
22,Coplin Joan,Baker Timothy
36,Crumpet Erica,Smith Tracy
7,Dare Nancy,Joplette Janice
28,Farrell David,null
13,Farrell Jemima,null


### Question 6
Produce a count of the number of recommendations each member has made. Order by member ID.

In [0]:
from pyspark.sql import functions as F
members = spark.table("members")

# Filter out the nulls (members who haven't recommended anyone)
# Group by the recommender's ID
# Count the occurrences and sort
result = members \
    .filter(F.col("recommendedby").isNotNull()) \
    .groupBy("recommendedby") \
    .count() \
    .orderBy("recommendedby")



result.show()

+-------------+-----+
|recommendedby|count|
+-------------+-----+
|            1|    5|
|            2|    3|
|            3|    1|
|            4|    2|
|            5|    1|
|            6|    1|
|            9|    2|
|           11|    1|
|           13|    2|
|           15|    1|
|           16|    1|
|           20|    1|
|           30|    1|
+-------------+-----+



### Question 7
Produce a list of the total number of slots booked per facility. For now, just produce an output table consisting of facility id and slots, sorted by facility id.


In [0]:
from pyspark.sql import functions as F

bookings = spark.table("bookings")

# Group by the facility ID
# Aggregate the sum of the 'slots' column
# Sort by facility ID
result = bookings \
    .groupBy("facid") \
    .agg(F.sum("slots").alias("Total Slots")) \
    .orderBy("facid")

result.show()

+-----+-----------+
|facid|Total Slots|
+-----+-----------+
|    0|       1320|
|    1|       1278|
|    2|       1209|
|    3|        830|
|    4|       1404|
|    5|        228|
|    6|       1104|
|    7|        908|
|    8|        911|
+-----+-----------+



### Question 8
Produce a list of the total number of slots booked per facility in the month of September 2012. Produce an output table consisting of facility id and slots, sorted by the number of slots.

In [0]:
from pyspark.sql import functions as F

# Filter for the month of September 2012
# Group by facility ID
# Sum the slots and rename the column
# Sort by the calculated "Total Slots"
result = bookings \
    .filter(
        (F.col("starttime") >= "2012-09-01") & 
        (F.col("starttime") < "2012-10-01")
    ) \
    .groupBy("facid") \
    .agg(F.sum("slots").alias("Total Slots")) \
    .orderBy("Total Slots")

result.show()

+-----+-----------+
|facid|Total Slots|
+-----+-----------+
|    5|        122|
|    3|        422|
|    7|        426|
|    8|        471|
|    6|        540|
|    2|        570|
|    1|        588|
|    0|        591|
|    4|        648|
+-----+-----------+



### Question 9
Produce a list of the total number of slots booked per facility per month in the year of 2012. Produce an output table consisting of facility id and slots, sorted by the id and month.

In [0]:
from pyspark.sql import functions as F

# Filter for the year 2012
# Extract the month into its own column (we use F.year and F.month to extract the year and month from the timestamp)
# In PySpark, if you want to use a transformed value in both your grouping and your final output, it is often easiest to create it as a temporary column first using withColumn.
# Group by both facility and month
# Sum the slots and sort
result = bookings \
    .filter(F.year("starttime") == 2012) \
    .withColumn("month", F.month("starttime")) \
    .groupBy("facid", "month") \
    .agg(F.sum("slots").alias("Total Slots")) \
    .orderBy("facid", "month")

result.show()

+-----+-----+-----------+
|facid|month|Total Slots|
+-----+-----+-----------+
|    0|    7|        270|
|    0|    8|        459|
|    0|    9|        591|
|    1|    7|        207|
|    1|    8|        483|
|    1|    9|        588|
|    2|    7|        180|
|    2|    8|        459|
|    2|    9|        570|
|    3|    7|        104|
|    3|    8|        304|
|    3|    9|        422|
|    4|    7|        264|
|    4|    8|        492|
|    4|    9|        648|
|    5|    7|         24|
|    5|    8|         82|
|    5|    9|        122|
|    6|    7|        164|
|    6|    8|        400|
+-----+-----+-----------+
only showing top 20 rows


### Question 10
Find the total number of members (including guests) who have made at least one booking.


In [0]:
from pyspark.sql import functions as F

# countDistinct function directly on the column
result = bookings.select(F.countDistinct("memid").alias("count"))

result.show()

+-----+
|count|
+-----+
|   30|
+-----+



### Question 11
Produce a list of each member name, id, and their first booking after September 1st 2012. Order by member ID.


In [0]:
from pyspark.sql import functions as F

# Join members and bookings on member id field
# Filter for bookings on or after September 1st
# Group by member info and find the minimum starttime
result = members.alias("m") \
    .join(bookings.alias("b"), F.col("m.memid") == F.col("b.memid")) \
    .filter(F.col("b.starttime") >= "2012-09-01") \
    .groupBy("m.surname", "m.firstname", "m.memid") \
    .agg(F.min("b.starttime").alias("min(starttime)")) \
    .orderBy("m.memid")

result.show()

+--------+---------+-----+-------------------+
| surname|firstname|memid|     min(starttime)|
+--------+---------+-----+-------------------+
|   GUEST|    GUEST|    0|2012-09-01 08:00:00|
|  Darren|    Smith|    1|2012-09-01 09:00:00|
|   Tracy|    Smith|    2|2012-09-01 11:30:00|
|     Tim|   Rownam|    3|2012-09-01 16:00:00|
|  Janice| Joplette|    4|2012-09-01 15:00:00|
|  Gerald|  Butters|    5|2012-09-02 12:30:00|
|  Burton|    Tracy|    6|2012-09-01 15:00:00|
|   Nancy|     Dare|    7|2012-09-01 12:30:00|
|     Tim|   Boothe|    8|2012-09-01 08:30:00|
|  Ponder| Stibbons|    9|2012-09-01 11:00:00|
| Charles|     Owen|   10|2012-09-01 11:00:00|
|   David|    Jones|   11|2012-09-01 09:30:00|
|    Anne|    Baker|   12|2012-09-01 14:30:00|
|  Jemima|  Farrell|   13|2012-09-01 09:30:00|
|    Jack|    Smith|   14|2012-09-01 11:00:00|
|Florence|    Bader|   15|2012-09-01 10:30:00|
| Timothy|    Baker|   16|2012-09-01 15:00:00|
|   David|   Pinker|   17|2012-09-01 08:30:00|
| Matthew|  G

### Question 12
Output the names of all members, formatted as 'Surname, Firstname'


In [0]:
from pyspark.sql import functions as F

# Use concat_ws to join surname and firstname with ', '
result = members.select(
    F.concat_ws(", ", F.col("surname"), F.col("firstname")).alias("name")
)

result.show()

+----------------+
|            name|
+----------------+
|    GUEST, GUEST|
|   Darren, Smith|
|    Tracy, Smith|
|     Tim, Rownam|
|Janice, Joplette|
| Gerald, Butters|
|   Burton, Tracy|
|     Nancy, Dare|
|     Tim, Boothe|
|Ponder, Stibbons|
|   Charles, Owen|
|    David, Jones|
|     Anne, Baker|
| Jemima, Farrell|
|     Jack, Smith|
| Florence, Bader|
|  Timothy, Baker|
|   David, Pinker|
|Matthew, Genting|
| Anna, Mackenzie|
+----------------+
only showing top 20 rows


### Question 13
Perform a case-insensitive search to find all facilities whose name begins with 'tennis'. Retrieve all columns.


In [0]:
from pyspark.sql import functions as F

facilities = spark.table("facilities")

# Take the name column and make it lowercase
# Check if it starts with 'tennis' using startswith("tennis")
# Select all columns (*)
result = facilities.filter(
    F.lower(F.col("name")).startswith("tennis")
)

result.show()

+-----+--------------+----------+---------+-------------+------------------+
|facid|          name|membercost|guestcost|initialoutlay|monthlymaintenance|
+-----+--------------+----------+---------+-------------+------------------+
|    0|Tennis Court 1|       5.0|     25.0|        10000|               200|
|    1|Tennis Court 2|       5.0|     25.0|         8000|               200|
+-----+--------------+----------+---------+-------------+------------------+



### Question 14
You've noticed that the club's member table has telephone numbers with very inconsistent formatting. You'd like to find all the telephone numbers that contain parentheses, returning the member ID and telephone number sorted by member ID.


In [0]:
from pyspark.sql import functions as F

# Filter the telephone column using a regex pattern [()]It tells Spark to look for any character that is either an opening parenthesis ( or a closing parenthesis ).
# rlike is like regex in SQL
# Select the specific columns
# Sort by member ID
result = members.filter(F.col("telephone").rlike("[()]")) \
    .select("memid", "telephone") \
    .orderBy("memid")

result.show()

+-----+--------------+
|memid|     telephone|
+-----+--------------+
|    0|(000) 000-0000|
|    3|(844) 693-0723|
|    4|(833) 942-4710|
|    5|(844) 078-4130|
|    6|(822) 354-9973|
|    7|(833) 776-4001|
|    8|(811) 433-2547|
|    9|(833) 160-3900|
|   10|(855) 542-5251|
|   11|(844) 536-8036|
|   13|(855) 016-0163|
|   14|(822) 163-3254|
|   15|(833) 499-3527|
|   20|(811) 972-1377|
|   21|(822) 661-2898|
|   22|(822) 499-2232|
|   24|(822) 413-1470|
|   27|(822) 989-8876|
|   28|(855) 755-9876|
|   29|(855) 894-3758|
+-----+--------------+
only showing top 20 rows


### Question 15
You'd like to produce a count of how many members you have whose surname starts with each letter of the alphabet. Sort by the letter, and don't worry about printing out a letter if the count is 0.


In [0]:
from pyspark.sql import functions as F

# Extract the first character of the surname. This works exactly like the SQL SUBSTR. We start at position 1 and take a length of 1
# Group by that character
# Count the occurrences and sort alphabetically
result = members \
    .withColumn("letter", F.substring(F.col("surname"), 1, 1)) \
    .groupBy("letter") \
    .count() \
    .orderBy("letter")

result.show()

+------+-----+
|letter|count|
+------+-----+
|     A|    2|
|     B|    1|
|     C|    1|
|     D|    6|
|     E|    1|
|     F|    1|
|     G|    2|
|     H|    3|
|     J|    5|
|     M|    2|
|     N|    1|
|     P|    1|
|     R|    1|
|     T|    4|
+------+-----+



In [0]:

from pyspark.sql import functions as F

# members.columns[1] selects the 2nd column regardless of its name as it is 2nd column in members table
surname_col = members.columns[1]

result = members \
    .select(F.substring(F.trim(F.col(surname_col)), 1, 1).alias("letter")) \
    .groupBy("letter") \
    .count() \
    .orderBy("letter")

result.show()

+------+-----+
|letter|count|
+------+-----+
|     B|    5|
|     C|    2|
|     D|    1|
|     F|    2|
|     G|    2|
|     H|    1|
|     J|    3|
|     M|    1|
|     O|    1|
|     P|    2|
|     R|    2|
|     S|    6|
|     T|    2|
|     W|    1|
+------+-----+



### Question 16
Produce a list of all the dates in October 2012. They can be output as a timestamp (with time set to midnight) or a date.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import expr

# Create a single-row DataFrame with the start and end of October. spark.range(1) just creates a dummy starting point so we have a row to work with.
# Use sequence() to generate an array of all dates in between
# Use explode() to turn that array into individual rows
result = spark.range(1) \
    .select(
        F.explode(
            F.sequence(
                F.to_date(F.lit("2012-10-01")), 
                F.to_date(F.lit("2012-10-31")), 
                expr("interval 1 day")
            )
        ).alias("ts")
    )

result.show(31)

#using F.sequence : At this stage, you still only have one row, but that row contains a massive list of 31 dates sitting inside a single cell.
#with F.explore : Instead of one row with a list of 31 dates, it creates 31 rows, each containing one date from that list.
# In PySpark, expr (short for expression) is a powerful bridge that allows you to write SQL-like syntax inside your Python code.

+----------+
|        ts|
+----------+
|2012-10-01|
|2012-10-02|
|2012-10-03|
|2012-10-04|
|2012-10-05|
|2012-10-06|
|2012-10-07|
|2012-10-08|
|2012-10-09|
|2012-10-10|
|2012-10-11|
|2012-10-12|
|2012-10-13|
|2012-10-14|
|2012-10-15|
|2012-10-16|
|2012-10-17|
|2012-10-18|
|2012-10-19|
|2012-10-20|
|2012-10-21|
|2012-10-22|
|2012-10-23|
|2012-10-24|
|2012-10-25|
|2012-10-26|
|2012-10-27|
|2012-10-28|
|2012-10-29|
|2012-10-30|
|2012-10-31|
+----------+



### Question 17
Return a count of bookings for each month, sorted by month


In [0]:
from pyspark.sql import functions as F

# Truncate the date to the beginning of the month (e.g., 2012-10-15 -> 2012-10-01)
# Group by that month-start date
# Count and sort
result = bookings \
    .withColumn("month", F.date_trunc("month", F.col("starttime"))) \
    .groupBy("month") \
    .count() \
    .orderBy("month")

#F.date_trunc: This keeps the year and month intact but "zeros out" the days and times. It turns every booking in October 2012 into 2012-10-01 00:00:00.
result.show()

+-------------------+-----+
|              month|count|
+-------------------+-----+
|2012-07-01 00:00:00|  658|
|2012-08-01 00:00:00| 1472|
|2012-09-01 00:00:00| 1913|
|2013-01-01 00:00:00|    1|
+-------------------+-----+

